# Sequence Model Adaptation Experiments (BiLSTM + Transformer)

Adaptation strategies on both sequence probes against the 2024–2026 AndroZoo hold-out:
1. **Baseline** — from saved prediction CSVs (no re-inference)
2. **Threshold Recalibration** — t=0.70 / t=0.80 applied to saved probabilities
3. **Few-Shot Fine-tune** — adapt from pre-trained checkpoint (5% / 10% of AndroZoo)
4. **Upper Bound** — fine-tune on 80% of AndroZoo, evaluate on 20%

In [1]:
import copy, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
# GPU with auto-fallback to CPU
if torch.cuda.is_available():
    try:
        torch.cuda.memory.set_per_process_memory_fraction(0.85)
        DEVICE = torch.device("cuda")
    except Exception:
        DEVICE = torch.device("cpu")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    free_mb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_allocated()) // (1024**2)
    print(f"  GPU free: {free_mb} MB")

Device: cuda
  GPU free: 3770 MB


In [3]:
ROOT        = Path("/home/tan/GitHub/paper-kangal-dynamic/dynamic_ieee_paper/deneyler")
DATA_ROOT   = ROOT.parent
AZ_DIR      = DATA_ROOT / "androzoo_dataset"
RESULT_ROOT = ROOT / "results"
ADAPT_DIR   = RESULT_ROOT / "adaptation"
SEQ_INFER   = RESULT_ROOT / "androzoo_sequence_inference"
SEQ_BUNDLE  = RESULT_ROOT / "androzoo_inference_model"
ADAPT_DIR.mkdir(parents=True, exist_ok=True)

N_PER_CLASS = 3_000
print("Paths OK")

Paths OK


## Utility Functions

In [4]:
def load_seqlogs(path):
    with open(path) as f:
        return [json.loads(l) for l in f]


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "macro_f1":      round(f1_score(y_true, y_pred, average="macro"), 4),
        "roc_auc":       round(roc_auc_score(y_true, y_prob), 4),
        "pr_auc":        round(average_precision_score(y_true, y_prob), 4),
        "fpr":           round(fp / (fp + tn + 1e-12), 4),
        "fnr":           round(fn / (fn + tp + 1e-12), 4),
        "benign_recall": round(tn / (tn + fp + 1e-12), 4),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

## Model Definitions

In [5]:
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size=64, embed_dim=96, hidden_dim=128,
                 num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 1),
        )

    def forward(self, x):
        emb = self.embedding(x)
        _, (h, _) = self.lstm(emb)
        feat = torch.cat([h[-2], h[-1]], dim=-1)
        return self.head(feat).squeeze(-1)


class TransformerModel(nn.Module):
    def __init__(self, vocab_size=64, embed_dim=96, num_layers=2,
                 nhead=4, dim_feedforward=256, max_seq_len=256, dropout=0.2):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_emb   = nn.Embedding(max_seq_len, embed_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        emb  = self.token_emb(x) + self.pos_emb(pos)
        out  = self.encoder(emb, src_key_padding_mask=(x == 0))
        return self.head(out.mean(dim=1)).squeeze(-1)


class SeqDataset(Dataset):
    def __init__(self, records, stoi, max_len=256, unk_id=1):
        self.records = records
        self.stoi    = stoi
        self.max_len = max_len
        self.unk_id  = unk_id

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec  = self.records[idx]
        tags = [e["tag"] for e in rec["seq_log"]][:self.max_len]
        ids  = [self.stoi.get(t, self.unk_id) for t in tags]
        ids  = ids + [0] * (self.max_len - len(ids))
        y    = 1 if rec["label"] == "malware" else 0
        return torch.tensor(ids, dtype=torch.long), torch.tensor(y, dtype=torch.float)


@torch.no_grad()
def seq_predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()
        probs.extend(p.tolist())
        labels.extend(y.numpy().tolist())
    return np.array(probs), np.array(labels)


def seq_fine_tune(model, train_ldr, val_ldr, lr=2e-5, n_epochs=5, patience=3):
    opt       = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    best_f1, best_state, no_imp = -1.0, None, 0
    for ep in range(1, n_epochs + 1):
        model.train()
        ep_loss = 0.0
        for x, y in train_ldr:
            opt.zero_grad()
            loss = criterion(model(x.to(DEVICE)), y.to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()
        if val_ldr:
            p, l = seq_predict(model, val_ldr)
            f1 = f1_score(l, (p >= 0.5).astype(int), average="macro")
            print(f"    ep {ep:2d}  loss={ep_loss/len(train_ldr):.4f}  val_F1={f1:.4f}", flush=True)
            if f1 > best_f1 + 1e-4:
                best_f1, best_state, no_imp = f1, copy.deepcopy(model.state_dict()), 0
            else:
                no_imp += 1
            if no_imp >= patience:
                print(f"    Early stop at epoch {ep}.")
                break
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    return model

print("Model definitions OK")

Model definitions OK


## Load AndroZoo Sequence Data

In [6]:
print("Loading AndroZoo sequence logs …")
az_benign  = load_seqlogs(AZ_DIR / "androzoo_benign_seqlogs.jsonl")[:N_PER_CLASS]
az_malware = load_seqlogs(AZ_DIR / "androzoo_malware_seqlogs.jsonl")[:N_PER_CLASS]
az_all     = az_benign + az_malware
rng        = np.random.RandomState(SEED)
idx_shuf   = rng.permutation(len(az_all))
az_all     = [az_all[i] for i in idx_shuf]
y_az       = np.array([1 if r["label"] == "malware" else 0 for r in az_all])
print(f"  {len(az_all)} sequences  malware={y_az.sum()}  benign={(y_az==0).sum()}")

Loading AndroZoo sequence logs …
  6000 sequences  malware=3000  benign=3000


## Model Configs

In [7]:
SEQ_CONFIGS = {
    "BiLSTM": {
        "ckpt":         RESULT_ROOT / "kronodroid_sequence_bilstm.pt",
        "model_cls":    BiLSTMModel,
        "model_kwargs": dict(vocab_size=64, embed_dim=96, hidden_dim=128,
                             num_layers=2, dropout=0.2),
        "pred_csv":     SEQ_INFER / "androzoo_sequence_predictions_bilstm.csv",
    },
    "Transformer": {
        "ckpt":         SEQ_BUNDLE / "androzoo_sequence_model_bundle.pt",
        "model_cls":    TransformerModel,
        "model_kwargs": dict(vocab_size=64, embed_dim=96, num_layers=2,
                             nhead=4, dim_feedforward=256, max_seq_len=256,
                             dropout=0.2),
        "pred_csv":     SEQ_INFER / "androzoo_sequence_predictions_transformer.csv",
    },
}
print("Configs loaded")

Configs loaded


## Adaptation Experiments

In [8]:
seq_rows = []

for seq_name, cfg in SEQ_CONFIGS.items():
    print(f"\n{'═'*60}")
    print(f"  {seq_name}")
    print(f"{'═'*60}")

    ckpt  = torch.load(cfg["ckpt"], map_location="cpu", weights_only=False)
    state = ckpt.get("model_state_dict", ckpt)
    stoi  = ckpt.get("stoi", None)
    if stoi is None:
        with open(SEQ_BUNDLE / "sequence_vocab.json") as f:
            stoi = json.load(f)

    # ── (1) Baseline ──────────────────────────────────────────────────────────
    pdf          = pd.read_csv(cfg["pred_csv"])
    pkg_to_prob  = dict(zip(pdf["package_name"], pdf["malware_probability"]))
    pkg_to_label = dict(zip(pdf["package_name"], pdf["label_bin"]))
    probs_base   = np.array([pkg_to_prob.get(r["package_name"], 0.5) for r in az_all])
    labels_base  = np.array([pkg_to_label.get(r["package_name"], y_az[i])
                              for i, r in enumerate(az_all)])

    m = compute_metrics(labels_base, probs_base)
    m.update({"model": seq_name, "strategy": "No adaptation (baseline)", "threshold": 0.5})
    seq_rows.append(m)
    print(f"  Baseline    F1={m['macro_f1']}  FPR={m['fpr']}  FNR={m['fnr']}")

    # ── (2) Threshold Recalibration ───────────────────────────────────────────
    for t in [0.70, 0.80]:
        m = compute_metrics(labels_base, probs_base, threshold=t)
        m.update({"model": seq_name, "strategy": f"Threshold recalib. (t={t:.2f})", "threshold": t})
        seq_rows.append(m)
        print(f"  Recal t={t}   F1={m['macro_f1']}  FPR={m['fpr']}  FNR={m['fnr']}")

    # ── (3) Few-Shot Fine-Tuning ──────────────────────────────────────────────
    for adapt_frac in [0.05, 0.10]:
        idx_adapt, idx_test = train_test_split(
            np.arange(len(az_all)),
            test_size=(1 - adapt_frac),
            stratify=y_az,
            random_state=SEED,
        )
        adapt_seqs = [az_all[i] for i in idx_adapt]
        test_seqs  = [az_all[i] for i in idx_test]
        n_adapt    = len(adapt_seqs)

        tr_ldr = DataLoader(SeqDataset(adapt_seqs, stoi), batch_size=32,
                            shuffle=True, num_workers=0)
        te_ldr = DataLoader(SeqDataset(test_seqs,  stoi), batch_size=64,
                            shuffle=False, num_workers=0)

        print(f"\n  Few-shot {int(adapt_frac*100)}%  (n_adapt={n_adapt}, test={len(test_seqs)})")
        model_ft = cfg["model_cls"](**cfg["model_kwargs"]).to(DEVICE)
        model_ft.load_state_dict(state)
        model_ft = seq_fine_tune(model_ft, tr_ldr, te_ldr, lr=2e-5, n_epochs=5, patience=3)

        prob_ft, label_ft = seq_predict(model_ft, te_ldr)
        m = compute_metrics(label_ft, prob_ft)
        m.update({
            "model": seq_name,
            "strategy": f"Few-shot fine-tune ({int(adapt_frac*100)}%, n={n_adapt})",
            "threshold": 0.5,
        })
        seq_rows.append(m)
        print(f"  → F1={m['macro_f1']}  FPR={m['fpr']}  FNR={m['fnr']}")
        del model_ft
        torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

    # ── (4) Upper Bound (80/20) ───────────────────────────────────────────────
    print(f"\n  Upper bound (80/20 fine-tune) …")
    idx_tr, idx_te = train_test_split(
        np.arange(len(az_all)), test_size=0.20, stratify=y_az, random_state=SEED,
    )
    tr_ldr_ub = DataLoader(SeqDataset([az_all[i] for i in idx_tr], stoi),
                           batch_size=64, shuffle=True, num_workers=0)
    te_ldr_ub = DataLoader(SeqDataset([az_all[i] for i in idx_te], stoi),
                           batch_size=64, shuffle=False, num_workers=0)

    model_ub = cfg["model_cls"](**cfg["model_kwargs"]).to(DEVICE)
    model_ub.load_state_dict(state)
    model_ub = seq_fine_tune(model_ub, tr_ldr_ub, te_ldr_ub, lr=1e-4, n_epochs=8, patience=3)

    prob_ub, label_ub = seq_predict(model_ub, te_ldr_ub)
    m = compute_metrics(label_ub, prob_ub)
    m.update({"model": seq_name, "strategy": "Full fine-tune (upper bound, 80/20)", "threshold": 0.5})
    seq_rows.append(m)
    print(f"  → F1={m['macro_f1']}  FPR={m['fpr']}  FNR={m['fnr']}")
    del model_ub
    torch.cuda.empty_cache() if DEVICE.type == "cuda" else None


════════════════════════════════════════════════════════════
  BiLSTM
════════════════════════════════════════════════════════════
  Baseline    F1=0.7987  FPR=0.208  FNR=0.1947
  Recal t=0.7   F1=0.7979  FPR=0.1793  FNR=0.2247
  Recal t=0.8   F1=0.7922  FPR=0.1603  FNR=0.2543

  Few-shot 5%  (n_adapt=300, test=5700)
    ep  1  loss=1.4621  val_F1=0.5899
    ep  2  loss=1.3425  val_F1=0.6007
    ep  3  loss=1.1902  val_F1=0.6110
    ep  4  loss=1.0572  val_F1=0.6218
    ep  5  loss=0.9784  val_F1=0.6542
  → F1=0.6542  FPR=0.1575  FNR=0.5123

  Few-shot 10%  (n_adapt=600, test=5400)
    ep  1  loss=1.5098  val_F1=0.6055
    ep  2  loss=1.2187  val_F1=0.6282
    ep  3  loss=1.0286  val_F1=0.6887
    ep  4  loss=0.8230  val_F1=0.7923
    ep  5  loss=0.6733  val_F1=0.7957
  → F1=0.7957  FPR=0.2559  FNR=0.1515

  Upper bound (80/20 fine-tune) …
    ep  1  loss=0.6746  val_F1=0.8458
    ep  2  loss=0.4241  val_F1=0.8625
    ep  3  loss=0.3552  val_F1=0.8750
    ep  4  loss=0.2971  val_F1=0.

## Save & Display Results

In [9]:
seq_df = pd.DataFrame(seq_rows)
out_csv = ADAPT_DIR / "adaptation_summary_sequence.csv"
seq_df.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")
seq_df[["model", "strategy", "macro_f1", "roc_auc", "fpr", "fnr"]]

Saved → /home/tan/GitHub/paper-kangal-dynamic/dynamic_ieee_paper/deneyler/results/adaptation/adaptation_summary_sequence.csv


,model,strategy,macro_f1,roc_auc,fpr,fnr
0,BiLSTM,No adaptation (baseline),0.7987,0.8366,0.2080,0.1947
1,BiLSTM,Threshold recalib. (t=0.70),0.7979,0.8366,0.1793,0.2247
2,BiLSTM,Threshold recalib. (t=0.80),0.7922,0.8366,0.1603,0.2543
3,BiLSTM,"Few-shot fine-tune (5%, n=300)",0.6542,0.8119,0.1575,0.5123
4,BiLSTM,"Few-shot fine-tune (10%, n=600)",0.7957,0.8582,0.2559,0.1515
5,BiLSTM,"Full fine-tune (upper bound, 80/20)",0.9099,0.9573,0.0650,0.1150
6,Transformer,No adaptation (baseline),0.7746,0.8523,0.1460,0.3020
7,Transformer,Threshold recalib. (t=0.70),0.7722,0.8523,0.1300,0.3213
8,Transformer,Threshold recalib. (t=0.80),0.7693,0.8523,0.1227,0.3337
9,Transformer,"Few-shot fine-tune (5%, n=300)",0.7647,0.8207,0.2677,0.2025
